# GSA Timetabling

This notebook implements the railway scheduling problem using the Gravitational Search Algorithm (GSA) based on the original GSA_M project.

## 0. Load Libraries

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import shutil
from pathlib import Path

from robin.supply.generator.entities import SupplyGenerator
from robin.supply.entities import Supply

from src.craft.entities import RevenueSimulator
from src.craft.gsa.entities import GSA, Boundaries, Solution

## 1. Generate Supply

In [ ]:
supply_config_path = Path("../configs/supply_generator/supply_data.yaml")
generator_config_path = Path("../configs/supply_generator/config.yaml")
generator_save_path = Path('../data/results/supply_gsa.yaml')

Path('../data/results').mkdir(parents=True, exist_ok=True)

generator = SupplyGenerator.from_yaml(
    path_config_supply=supply_config_path,
    path_config_generator=generator_config_path
)

generator.generate(
    n_services=25,
    output_path=generator_save_path,
    seed=42,
    progress_bar=True,
    without_conflicts=False
)

print(f'Generated {len(generator.services)} services')

## 2. Load Supply and Generate Revenue Behavior

In [ ]:
supply = Supply.from_yaml(path=str(generator_save_path))
print(f'Loaded {len(supply.services)} services')

revenue_simulator = RevenueSimulator(supply=supply)
revenue_behavior = revenue_simulator.simulate_revenue(alpha=2/3)

print(f'Revenue behavior computed for {len(revenue_behavior)} services')

## 3. Initialize Timetabling Problem

In [ ]:
from src.craft.mealpy.entities import MealpyTimetabling

timetabling = MealpyTimetabling(
    requested_services=supply.services,
    revenue_behavior=revenue_behavior,
    safe_headway=10,
    max_stop_time=10
)

print(f'Problem initialized with {timetabling.n_services} services')
print(f'Real variables: {len(timetabling.boundaries.real)}')
print(f'Discrete variables: {len(timetabling.boundaries.discrete)}')

## 4. Define Objective Function for GSA

## 5. Run GSA Optimization

## 6. Results

In [ ]:
best_solution = gsa.solution_history[-1]
print(f"Best fitness: {history['Fitness'].iloc[-1]}")
print(f"Best solution real variables: {len(best_solution.real)}")
print(f"Best solution discrete: {best_solution.discrete}")

history.tail(10)

## 7. Update Supply with Solution

In [ ]:
solution = Solution(real=best_solution.real, discrete=best_solution.discrete)
updated_services = timetabling.update_supply(str(generator_save_path), solution)

print(f'Updated supply contains {len(updated_services)} services')

In [ ]:
# Plot convergence
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(history['Fitness'])
plt.xlabel('Iteration')
plt.ylabel('Fitness')
plt.title('GSA Convergence')
plt.grid(True)
plt.show()